# BEAM Linkstats Consist Analysis

This notebook bootstraps a Consist tracker against an **archived PILATES run** (scratch storage), lists linkstats artifacts by facets, and computes first-pass summary + delta metrics over phys-sim iterations.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("Kernel Python:", sys.executable)


In [ ]:
# --- Configure these for your environment/run ---
USER = os.environ.get("USER", "$USER")
RUN_NAME = "new-asim-consist-flash-20260209-103617"
SLURM_JOB_ID = os.environ.get("SLURM_JOB_ID", "")

OUTPUT_ROOT = Path(f"/global/scratch/users/{USER}/pilates-output").resolve()
PROJECT_ROOT = Path(f"/global/scratch/users/{USER}/sources/PILATES").resolve()
ARCHIVE_RUN_DIR = OUTPUT_ROOT / RUN_NAME

# DB path resolution order:
# 1) PILATES_DB_PATH env override
# 2) node-local default (/local/job$SLURM_JOB_ID/db/pilates-analysis.duckdb)
# 3) archived run DB
DB_PATH_ENV = os.environ.get("PILATES_DB_PATH")
LOCAL_DB_PATH = Path(f"/local/job{SLURM_JOB_ID}/db/pilates-analysis.duckdb") if SLURM_JOB_ID else None
ARCHIVE_DB_PATH = ARCHIVE_RUN_DIR / "provenance.duckdb"

if DB_PATH_ENV:
    DB_PATH = Path(DB_PATH_ENV).expanduser()
elif LOCAL_DB_PATH is not None and LOCAL_DB_PATH.exists():
    DB_PATH = LOCAL_DB_PATH
else:
    DB_PATH = ARCHIVE_DB_PATH

TARGET_YEAR = 2018

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARCHIVE_RUN_DIR:", ARCHIVE_RUN_DIR)
print("LOCAL_DB_PATH:", LOCAL_DB_PATH)
print("DB_PATH:", DB_PATH)


In [ ]:
from pilates.utils.consist_analysis import (
    assign_effective_beam_sub_iteration,
    create_analysis_tracker,
    find_linkstats_artifacts,
    print_duckdb_health,
    summarize_linkstats_artifacts,
    summarize_linkstats_traveltime_deltas,
    summarize_linkstats_traveltime_deltas_hourly_weighted,
)

db_health = print_duckdb_health(db_path=DB_PATH, probe_open=True)

tracker = create_analysis_tracker(
    db_path=DB_PATH,
    archive_run_dir=ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    output_root=OUTPUT_ROOT,
)

print("Tracker mounts:")
for name, root in tracker.mounts.items():
    print(f"  {name}:// -> {root}")


In [ ]:
# Primary query: phys-sim unmodified linkstats parquet artifacts for one year across all outer iterations
artifacts_df = find_linkstats_artifacts(
    tracker,
    year=TARGET_YEAR,
    artifact_family="linkstats_unmodified_phys_sim_iter_parquet",
    namespace="beam",
    limit=20000,
)

print(f"Found {len(artifacts_df)} artifacts")
artifacts_df.head(20)

In [ ]:
# Quick facet/view sanity check
if artifacts_df.empty:
    print("No linkstats artifacts found with the requested filters.")
else:
    display(
        artifacts_df[[
            "key",
            "container_uri",
            "facet_source",
            "phys_sim_iteration",
            "beam_sub_iteration",
        ]].head(25)
    )
    print("Facet source summary:")
    display(artifacts_df["facet_source"].value_counts(dropna=False))


In [ ]:
# Keep only indexed snapshots and derive an effective sub-iteration index.
# Promoted final sub-iteration rows (beam_sub_iteration = NaN) are mapped to
# max(non-null sub-iteration) + 1 within each (run_id, year, iteration, phys_sim_iteration).
artifacts_clean = artifacts_df[artifacts_df["facet_source"] == "artifact_kv"].copy()
artifacts_clean = assign_effective_beam_sub_iteration(artifacts_clean)

# Keep raw value for debugging, but use effective value for analysis grouping.
artifacts_clean["beam_sub_iteration_raw"] = artifacts_clean["beam_sub_iteration"]
artifacts_clean["beam_sub_iteration"] = artifacts_clean["beam_sub_iteration_effective"]

# Optional logical de-dup in case multiple keys map to the same snapshot.
artifacts_clean = (
    artifacts_clean
    .sort_values(["run_id", "year", "iteration", "phys_sim_iteration", "beam_sub_iteration", "key"])
    .drop_duplicates(
        subset=["run_id", "year", "iteration", "phys_sim_iteration", "beam_sub_iteration"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(f"Raw artifacts: {len(artifacts_df)}")
print(f"Filtered artifacts: {len(artifacts_clean)}")

# Use volume-weighted travel time in per-artifact summary stats.
summary_df = summarize_linkstats_artifacts(
    artifacts_clean,
    tracker=tracker,
    traveltime_weighting="volume_weighted",
)
print(f"Computed summaries for {len(summary_df)} artifacts via Consist views")
summary_df.head(20)


In [ ]:
# First-pass rollup by sub-iteration + phys-sim iteration
if summary_df.empty:
    print("No summary rows available. Check mounts/path resolution above.")
else:
    rollup = (
        summary_df.groupby(["year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], dropna=False)
        .agg(
            artifacts=("key", "count"),
            rows_total=("row_count", "sum"),
            distinct_links_mean=("distinct_links", "mean"),
            volume_sum_total=("volume_sum", "sum"),
            vmt_miles_total=("vmt_miles", "sum"),
            vht_hours_total=("vht_hours", "sum"),
            total_delay_hours_total=("total_delay_hours", "sum"),
            vmt_per_vht_mph_mean=("vmt_per_vht_mph", "mean"),
            vht_per_vmt_hours_per_mile_mean=("vht_per_vmt_hours_per_mile", "mean"),
            traveltime_mean_mean=("traveltime_mean", "mean"),
            traveltime_p95_mean=("traveltime_p95", "mean"),
        )
        .assign(
            vmt_per_vht_mph_total=lambda df: df["vmt_miles_total"] / df["vht_hours_total"],
            vht_per_vmt_hours_per_mile_total=lambda df: df["vht_hours_total"] / df["vmt_miles_total"],
        )
        .reset_index()
        .sort_values(["year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], na_position="last")
    )
    display(rollup)


In [ ]:
# Consecutive phys-sim travel-time deltas within each (year, iteration, beam_sub_iteration) sequence
# Set USE_HOURLY_WEIGHTED = True to run the faster, hourly volume-weighted variant.
USE_HOURLY_WEIGHTED = True

if USE_HOURLY_WEIGHTED:
    delta_df = summarize_linkstats_traveltime_deltas_hourly_weighted(
        summary_df,
        tracker=tracker,
        exclude_zero_volume=True,
    )
    print(f"Computed {len(delta_df)} hourly-weighted travel-time delta rows")
else:
    delta_df = summarize_linkstats_traveltime_deltas(summary_df, tracker=tracker)
    print(f"Computed {len(delta_df)} travel-time delta rows")

delta_df


In [ ]:
# Plots
if delta_df.empty:
    print("delta_df is empty; skipping delta plots.")
else:
    sns.relplot(
        data=delta_df,
        x="phys_sim_iteration_prev",
        y="traveltime_delta_abs_mean",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )

if "rollup" in globals() and not rollup.empty:
    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="traveltime_mean_mean",
        hue="iteration",
        col="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="vmt_miles_total",
        hue="iteration",
        col="beam_sub_iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )

    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=rollup,
        x="vmt_miles_total",
        y="traveltime_mean_mean",
        hue="beam_sub_iteration",
        style="iteration",
        s=70,
    )
    plt.title("Travel Time vs VMT by Iteration/Sub-Iteration")
    plt.tight_layout()

if not delta_df.empty and {"prev_volume_total", "curr_volume_total"}.issubset(delta_df.columns):
    melt_df = delta_df.melt(
        id_vars=["iteration", "beam_sub_iteration", "phys_sim_iteration_prev"],
        value_vars=["prev_volume_total", "curr_volume_total"],
        var_name="period",
        value_name="volume_total",
    )
    sns.relplot(
        data=melt_df,
        x="phys_sim_iteration_prev",
        y="volume_total",
        hue="period",
        col="beam_sub_iteration",
        row="iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )


## Notes

- This notebook intentionally mounts `workspace://` to the archived run directory.
- It computes metrics from a single Consist grouped hybrid view (`tracker.create_grouped_view(...)`) rather than direct parquet paths.
- If view creation fails, verify mounts and ensure your Consist DB schema matches your Consist package version.
- You can switch artifact family to `linkstats_parquet` or `linkstats` in `find_linkstats_artifacts(...)` for other analyses.
